# Task 4: Context-Aware Chatbot Using LangChain / RAG

## Problem Statement & Objective
Build a conversational chatbot that **remembers context across turns** and **retrieves external information** from a document store during conversations, using Retrieval-Augmented Generation (RAG).

**Dataset:** a small custom knowledge base (`knowledge_base/*.txt`) describing this internship program — any `.txt` file dropped into that folder is automatically indexed.

In [1]:
from rag_chatbot import RagChatbot, load_documents

docs = load_documents()
print(f"Loaded {len(docs)} source document(s) from knowledge_base/")
print(docs[0].page_content[:400])

Loaded 1 source document(s) from knowledge_base/
DevelopersHub Corporation - Internal Knowledge Base (Sample Document)

About DevelopersHub Corporation
DevelopersHub Corporation is a technology company that runs an AI/ML Engineering Internship
program. The internship gives interns hands-on experience with transformer models, ML
pipelines, multimodal learning, conversational AI, and LLM applications.

Internship Structure
Interns are required to 


## Preprocessing: Chunking + Embedding + Vector Store

Documents are split into ~500-character overlapping chunks (`RecursiveCharacterTextSplitter`), embedded with a local `sentence-transformers/all-MiniLM-L6-v2` model, and indexed in a local **FAISS** vector store — no API key required.

## Model Development: Retrieval-Augmented Generation

`RagChatbot` retrieves the top-3 most relevant chunks per query from FAISS, builds a prompt containing that context plus the recent conversation history, and generates the answer with a local `google/flan-t5-base` sequence-to-sequence model. Context memory is implemented by keeping the last few (question, answer) turns and folding them into the prompt.

In [2]:
bot = RagChatbot()
print("Chatbot ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Chatbot ready.


## Evaluation: Multi-turn Conversation Demo

We ask a grounded question, then a **follow-up** question that depends on conversational context, to demonstrate both retrieval accuracy and context memory.

In [3]:
questions = [
    "What are the submission requirements for GitHub?",
    "What technologies are used in the internship?",
    "How many of those tasks must interns complete?",
]

transcript = []
for q in questions:
    result = bot.ask(q)
    transcript.append((q, result["answer"], [d.page_content[:120] for d in result["source_documents"]]))
    print(f"Q: {q}\nA: {result['answer']}\n")

Q: What are the submission requirements for GitHub?
A: Each completed task must be uploaded to a GitHub repository and submitted via Google Classroom.



Q: What technologies are used in the internship?
A: Hugging Face Transformers for NLP tasks, scikit-learn for classical ML pipelines, LangChain for retrieval-augmented generation and conversational AI, Streamlit and Gradio for lightweight model deployment, CNNs for image feature extraction, and joblib for exporting trained pipelines



Q: How many of those tasks must interns complete?
A: at least 3 out of 5 advanced tasks



## Visualizations: Retrieved Sources Per Turn

In [4]:
import pandas as pd

transcript_df = pd.DataFrame(transcript, columns=["question", "answer", "retrieved_source_snippets"])
transcript_df

,question,answer,retrieved_source_snippets
0,What are the submission requirements for GitHub?,Each completed task must be uploaded to a GitH...,[Submission Requirements\nEach completed task ...
1,What technologies are used in the internship?,"Hugging Face Transformers for NLP tasks, sciki...",[Tools and Technologies\nThe internship uses H...
2,How many of those tasks must interns complete?,at least 3 out of 5 advanced tasks,[Internship Structure\nInterns are required to...


## Final Summary / Insights

- The third question ("How many of those tasks must interns complete?") only makes sense with the prior turns' context ("those tasks" = the five advanced tasks mentioned earlier) — the chatbot resolves it correctly because `RagChatbot` folds recent history into the generation prompt.
- Using fully local, free models (MiniLM embeddings + FLAN-T5 generation) means the chatbot runs with no API key and no per-query cost, at some cost to answer fluency versus a larger hosted LLM.
- Returning `source_documents` alongside each answer makes the RAG pipeline auditable: every answer can be traced back to the exact chunk of the knowledge base it came from.
- For a production deployment, see `app_streamlit.py` for a full chat UI wrapping this same `RagChatbot` class.